In [ ]:
import sys
import math

# Please do not remove package declarations because these are used by the autograder.

# Insert your profile_hmm_with_pseudocounts function here, along with any subroutines you need
import sys
from collections import defaultdict

def legal_transitions(s,n_match) -> list[str]:
        if s == 'S' or s == 'I0':
            if n_match == 0:
                return ['I0', 'E']
            return ['I0', 'M1', 'D1']
        if s == 'E':
            return []
        kind = s[0]
        k    = int(s[1:])
        if kind in ('M', 'D'):
            if k == n_match:
                return [f'I{k}', 'E']
            return [f'I{k}', f'M{k+1}', f'D{k+1}']
        # kind == 'I'
        if k == n_match:
            return [f'I{k}', 'E']
        return [f'I{k}', f'M{k+1}', f'D{k+1}']

def apply_pseudo_row(counts, options, n_opts, sigma) -> list[float]:
        total = sum(counts.get(o, 0) for o in options)
        result = []
        for o in options:
            raw_cnt = counts.get(o, 0)
            # add pseudocount and normalise the counts 0,0.9,0.1 add 0.01 to everything and add them so the sum is 1.03. now divide everything by 1.03 to get the new normalized counts
            p_raw   = (raw_cnt / total) if total > 0 else (1.0 / n_opts)
            result.append((p_raw + sigma) / (1 + n_opts * sigma))
        return result

def apply_pseudo_emit(counts, alphabet,sigma,n_alph) -> list[float]:
    total = sum(counts.get(a, 0) for a in alphabet)
    result = []
    for a in alphabet:
        raw_cnt = counts.get(a, 0)
        p_raw   = (raw_cnt / total) if total > 0 else (1.0 / n_alph)
        result.append((p_raw + sigma) / (1 + n_alph * sigma))
    return result


def profile_hmm_with_pseudocounts(theta: float, sigma: float,
                                  alphabet: list[str],
                                  alignment: list[str]) -> tuple[list[str], list[str], list[list[float]], list[list[float]]]:

    n_seq  = len(alignment)
    n_col  = len(alignment[0])
    n_alph = len(alphabet)
    sym    = {c: i for i, c in enumerate(alphabet)} # alphabets and their orders

    is_match  = [sum(seq[c] == '-' for seq in alignment) / n_seq < theta
                 for c in range(n_col)]
    n_match   = sum(is_match)

 
    states    = ['S', 'I0']
    for i in range(1, n_match + 1):
        states += [f'M{i}', f'D{i}', f'I{i}']
    states.append('E')
    n_states  = len(states)
    sidx      = {s: i for i, s in enumerate(states)}

    trans_counts = defaultdict(lambda: defaultdict(float))
    emit_counts  = defaultdict(lambda: defaultdict(float))

    for seq in alignment:
        path = ['S']
        cur_match = 0
        for c in range(n_col):
            ch = seq[c]
            if is_match[c]:
                cur_match += 1
                path.append(f'D{cur_match}' if ch == '-' else f'M{cur_match}')
            else:
                if ch != '-':
                    path.append(f'I{cur_match}')
        path.append('E')

        for i in range(len(path) - 1):
            trans_counts[path[i]][path[i+1]] += 1
        
        #now the dict of dict gives us 
        # pair states with emitted characters
        col = 0
        for st in path[1:-1]:
            if st.startswith('M'):
                while not is_match[col]:
                    col += 1
                emit_counts[st][seq[col]] += 1
                col += 1
            elif st.startswith('D'):
                while not is_match[col]:
                    col += 1
                col += 1   # gap, no emission
            else:  # I
                while col < n_col and is_match[col]:
                    col += 1
                emit_counts[st][seq[col]] += 1
                col += 1

    transition = [[0.0] * n_states for _ in range(n_states)]
    emission   = [[0.0] * n_alph   for _ in range(n_states)]

    for s in states:
        si   = sidx[s]
        opts = legal_transitions(s,n_match)
        if not opts:
            continue
        probs = apply_pseudo_row(trans_counts[s], opts, len(opts),sigma)
        for o, p in zip(opts, probs):
            transition[si][sidx[o]] = p

        # Emissions only for non-silent states (M and I states)
        if s[0] in ('M', 'I'):
            ep = apply_pseudo_emit(emit_counts[s],alphabet,sigma,n_alph)
            for j, p in enumerate(ep):
                emission[si][j] = p

    return states, alphabet, transition, emission



    

In [ ]:
# Profile HMM with Pseudocounts Problem: Construct a profile HMM with pseudocounts from a multiple alignment.

# Input: A multiple alignment Alignment, a threshold value θ, and a pseudocount value σ.
# Output: HMM(Alignment, θ), in the form of transition and emission matrices.
# Code Challenge: Solve the Profile HMM with Pseudocounts Problem.

# Input: A threshold θ and a pseudocount σ, followed by an alphabet Σ, followed by a multiple alignment Alignment whose strings are formed from Σ.
# Output: The transition and emission matrices of HMM(Alignment, θ, σ).
# Note: The rows and columns of the transition matrix and the emission matrix must correspond exactly to the order in which you output your states and alphabet.

# Debug Datasets

# Sample Input 1:

# 0.358 0.01    
# --------
# A B C D E
# --------
# A-A
# ADA
# ACA
# A-C
# -EA
# D-A
# Sample Output 1:

# S I0 M1 D1 I1 M2 D2 I2 E
# --------
# A B C D E
# --------
# 	S	I0	M1	D1	I1	M2	D2	I2	E
# S	0	0.01	0.819	0.172	0	0	0	0	0
# I0	0	0.333	0.333	0.333	0	0	0	0	0
# M1	0	0	0	0	0.398	0.592	0.01	0	0
# D1	0	0	0	0	0.981	0.01	0.01	0	0
# I1	0	0	0	0	0.01	0.981	0.01	0	0
# M2	0	0	0	0	0	0	0	0.01	0.99
# D2	0	0	0	0	0	0	0	0.5	0.5
# I2	0	0	0	0	0	0	0	0.5	0.5
# E	0	0	0	0	0	0	0	0	0
# --------
# 	A	B	C	D	E
# S	0	0	0	0	0
# I0	0.2	0.2	0.2	0.2	0.2
# M1	0.771	0.01	0.01	0.2	0.01
# D1	0	0	0	0	0
# I1	0.01	0.01	0.327	0.327	0.327
# M2	0.803	0.01	0.168	0.01	0.01
# D2	0	0	0	0	0
# I2	0.2	0.2	0.2	0.2	0.2
# E	0	0	0	0	0
# Sample Input 2:

# 0.1 0.01
# --------
# A
# --------
# A
# Sample Output 2:

# S I0 M1 D1 I1 E
# --------
# A
# --------
# 	S	I0	M1	D1	I1	E
# S	0	0.01	0.981	0.01	0	0
# I0	0	0.333	0.333	0.333	0	0
# M1	0	0	0	0	0.01	0.99
# D1	0	0	0	0	0.5	0.5
# I1	0	0	0	0	0.5	0.5
# E	0	0	0	0	0	0
# --------
# 	A
# S	0
# I0	1.0
# M1	1.0
# D1	0
# I1	1.0
# E	0
# Sample Input 3:

# 0.1 0.01
# --------
# A B
# --------
# A
# Sample Output 3:

# S I0 M1 D1 I1 E
# --------
# A B
# --------
# 	S	I0	M1	D1	I1	E
# S	0	0.01	0.981	0.01	0	0
# I0	0	0.333	0.333	0.333	0	0
# M1	0	0	0	0	0.01	0.99
# D1	0	0	0	0	0.5	0.5
# I1	0	0	0	0	0.5	0.5
# E	0	0	0	0	0	0
# --------
# 	A	B
# S	0	0
# I0	0.5	0.5
# M1	0.99	0.01
# D1	0	0
# I1	0.5	0.5
# E	0	0